# BSAI – 5th Semester | Machine Learning
## LAB 3: Model Validation and K-Fold Cross-Validation
### Student Pass/Fail Classification using Logistic Regression

| Component | Description |
| :--- | :--- |
| **Dataset** | UCI Student Performance – Mathematics dataset (clean CSV) |
| **Model** | Logistic Regression |
| **Target** | Student Pass / Fail |
| **Validation** | 5-Fold Stratified Cross-Validation |
| **Model selection** | Logistic Regression hyperparameter C |

**Basis:** This manual follows the final W3 Colab uploaded by the instructor; the original code/results are preserved and student tasks are added.

---

### Prerequisites & Asset Setup
- **Required File:** `student-mat-clean.csv`
- **Repository Path:** This file is located in the repository under `Lab-03/assets/student-mat-clean.csv` (or `./assets/student-mat-clean.csv`). Ensure this CSV is placed in the same working directory as this notebook before executing Part 1, or modify `pd.read_csv()` to target your assets folder.

## 1. Learning Objectives
- Load and inspect the cleaned student dataset.
- Create a binary Pass/Fail target.
- Split data into training and testing sets.
- Train and evaluate Logistic Regression.
- Interpret accuracy, confusion matrix, precision, recall, and F1-score.
- Apply 5-Fold Stratified Cross-Validation.
- Compare hyperparameter values using mean CV accuracy.
- Select $C$ using K-Fold validation.
- Retrain the selected configuration on all training data and test once on the untouched test set.
- Explain why K-Fold supports model selection rather than directly increasing accuracy.

## 2. Dataset and Problem Definition
The lab uses `student-mat-clean.csv`. The final Colab run contains 395 students and 33 columns.

Create the binary target:
```python
df["Pass"] = (df["FinalGrade"] >= 10).astype(int)
```

| Value | Meaning |
| :--- | :--- |
| 0 | Fail |
| 1 | Pass |

**Final Colab distribution:** 265 Pass (67.09%) and 130 Fail (32.91%).

> **Important:** `FinalGrade` is not included in $X$ because `Pass` is derived directly from it. `G1` and `G2` are also excluded from the selected feature list in this lab.

## 3. Part 1 – Load and Prepare the Data
### 3.1 Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

### 3.2 Load the dataset

In [ ]:
df = pd.read_csv("student-mat-clean.csv")

print("Dataset shape:", df.shape)
display(df.head())
print("Columns:", df.columns.tolist())

### 3.3 Create Pass/Fail

| Class | Students | Percentage |
| :--- | :--- | :--- |
| Pass (1) | 265 | 67.09% |
| Fail (0) | 130 | 32.91% |

In [ ]:
df["Pass"] = (df["FinalGrade"] >= 10).astype(int)

display(df[["FinalGrade", "Pass"]].head(15))

print("0 = Fail")
print("1 = Pass")
print("\nNumber of students:")
print(df["Pass"].value_counts())
print("\nPercentage:")
print(df["Pass"].value_counts(normalize=True) * 100)

### 3.4 Select features

In [ ]:
features = [
    "age", "Medu", "Fedu", "studytime",
    "failures", "famrel", "freetime", "goout",
    "Dalc", "Walc", "health", "absences"
]

X = df[features]
y = df["Pass"]

print("X shape:", X.shape)
print("y shape:", y.shape)

### 3.5 Train-test split

| Data | Samples |
| :--- | :--- |
| Training / development | 316 |
| Untouched test | 79 |

> **Important:** K-Fold is performed on the 316 training/development observations. The 79 test observations remain untouched until final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

### Task 1: Dataset and Target Inspection
**Run Part 1 and record the dataset shape, number of features, and Pass/Fail counts.**

1. **Why is `FinalGrade` not included in $X$?**
   - **Answer:** Including `FinalGrade` causes severe target leakage. Because the binary label `Pass` is mathematically defined directly as `(FinalGrade >= 10)`, including `FinalGrade` in $X$ gives the model the ground truth answer directly, destroying realistic evaluation.

2. **What do Fail = 0 and Pass = 1 mean?**
   - **Answer:** `Fail = 0` designates students who scored below the passing threshold of 10 points on the final exam. `Pass = 1` designates students who achieved a passing score of 10 or greater (out of 20).

3. **Why do we keep a separate test set?**
   - **Answer:** To provide an unbiased, gold-standard evaluation of the final model's real-world generalization performance on completely unseen observations, ensuring hyperparameter tuning does not overfit validation data.

## 4. Part 2 – Baseline Logistic Regression
Train a baseline model first. This provides a reference point before K-Fold model selection.

### 4.1 Train and predict

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("First 20 predictions:", y_pred[:20])

### 4.2 Accuracy

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("Accuracy: {:.2f}%".format(accuracy * 100))

### 4.3 Confusion matrix

| Actual / Predicted | Fail | Pass |
| :--- | :--- | :--- |
| **Fail** | 7 (TN) | 19 (FP) |
| **Pass** | 8 (FN) | 45 (TP) |

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Fail", "Pass"]
)
disp.plot()
plt.title("Student Pass/Fail - Confusion Matrix")
plt.show()

### 4.4 Classification report

| Class/Metric | Precision | Recall | F1 | Support |
| :--- | :--- | :--- | :--- | :--- |
| **Fail** | 0.47 | 0.27 | 0.34 | 26 |
| **Pass** | 0.70 | 0.85 | 0.77 | 53 |
| **Accuracy** | | | 0.66 | 79 |
| **Macro avg** | 0.58 | 0.56 | 0.56 | 79 |
| **Weighted avg** | 0.63 | 0.66 | 0.63 | 79 |

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Fail", "Pass"]
    )
)

### Task 2: Baseline Model Interpretation
**Use the baseline confusion matrix and classification report to interpret the model.**

1. **Identify TP, TN, FP, and FN:**
   - True Positives (TP): **45** (Pass students predicted correctly as Pass)
   - True Negatives (TN): **7** (Fail students predicted correctly as Fail)
   - False Positives (FP): **19** (Fail students incorrectly predicted as Pass)
   - False Negatives (FN): **8** (Pass students incorrectly predicted as Fail)

2. **How many actual Pass students were predicted as Fail?**
   - **8 students** (False Negatives).

3. **How many actual Fail students were predicted as Pass?**
   - **19 students** (False Positives).

4. **Which class has higher recall?**
   - The **Pass** class has substantially higher recall (**0.85** vs. **0.27** for Fail).

5. **Why is accuracy alone not enough?**
   - Because of class imbalance (67.09% Pass vs 32.91% Fail). A naive model could attain ~67% accuracy simply by predicting Pass for everyone, while failing completely on the minority class. Here, 65.82% overall accuracy hides that 73% (19/26) of failing students went undetected.

## 5. Part 3 – 5-Fold Cross-Validation
K-Fold gives multiple validation views of the training/development data. With $K = 5$, each observation is used for validation once and for training in the other four folds.

### 5.1 Create five folds & 5.2 Inspect fold sizes

| Fold | Training | Validation |
| :--- | :--- | :--- |
| 1 | 252 | 64 |
| 2 | 253 | 63 |
| 3 | 253 | 63 |
| 4 | 253 | 63 |
| 5 | 253 | 63 |

In [ ]:
from sklearn.model_selection import StratifiedKFold

k = 5
skf = StratifiedKFold(
    n_splits=k,
    shuffle=True,
    random_state=42
)

fold = 1
for train_index, val_index in skf.split(X_train, y_train):
    print(f"Fold {fold}")
    print("Training samples:", len(train_index))
    print("Validation samples:", len(val_index))
    print("-------------------------")
    fold += 1

### 5.3 Train and validate each fold

| Fold | Validation Accuracy |
| :--- | :--- |
| 1 | 0.672 |
| 2 | 0.683 |
| 3 | 0.683 |
| 4 | 0.714 |
| 5 | 0.794 |

> **Important:** Do not choose Fold 5 as the final model just because it has the highest score. K-Fold uses all folds to obtain a more reliable validation view and to support model comparison.

In [ ]:
fold_scores = []
fold = 1

for train_index, val_index in skf.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_index]
    y_fold_train = y_train.iloc[train_index]

    X_fold_val = X_train.iloc[val_index]
    y_fold_val = y_train.iloc[val_index]

    model = LogisticRegression(max_iter=1000)
    model.fit(X_fold_train, y_fold_train)

    y_fold_pred = model.predict(X_fold_val)
    score = accuracy_score(y_fold_val, y_fold_pred)

    fold_scores.append(score)
    print(f"Fold {fold} Validation_Accuracy: {score:.3f}")
    fold += 1

print("\nMean Fold Accuracy: {:.3f}".format(np.mean(fold_scores)))

### Task 3: Understanding K-Fold
**Use the five fold accuracies to explain what K-Fold adds to the experiment.**

1. **Which fold is highest and which is lowest?**
   - Highest: **Fold 5** (0.794)
   - Lowest: **Fold 1** (0.672)

2. **Calculate the mean of the five accuracies:**
   $$\text{Mean} = \frac{0.672 + 0.683 + 0.683 + 0.714 + 0.794}{5} = \mathbf{0.709} \text{ (70.92\%)}$$

3. **Why should we not simply select the highest-scoring fold?**
   - A single high-scoring fold is usually an artifact of an unusually easy validation sample split rather than superior model learning. Selecting it introduces selection bias and will fail to generalize to realistic unseen data.

4. **Explain: 'A single validation split gives one view; K-Fold gives multiple views.'**
   - A single split produces high variance depending entirely on random seed luck. K-Fold partitions the entire development set so every single instance is validated exactly once, offering a complete and stable perspective across heterogeneous subsets.

## 6. Part 4 – K-Fold for Model Selection
### 6.1 Learning objective & What is C?
$C$ is the inverse regularization strength parameter ($C = 1 / \lambda$). Regularization penalizes overly large weights to keep the model from overfitting.
- **Small $C$:** Stronger regularization (forces weights closer to 0, simpler decision boundary).
- **Large $C$:** Weaker regularization (model fits closer to the training data, higher complexity).

*Note: $C$ is not the learning rate.*

### 6.2 Establish the baseline

In [ ]:
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, baseline_pred)

print("Baseline Test Accuracy:", round(baseline_accuracy, 3))
print("Baseline Test Accuracy: {:.2f}%".format(baseline_accuracy * 100))

### 6.3 Compare C values with 5-Fold CV

| C | Mean CV Accuracy | Std |
| :--- | :--- | :--- |
| 0.01 | 0.693 | 0.025 |
| 0.1 | 0.718 | 0.046 |
| 1 | 0.709 | 0.045 |
| 10 | 0.709 | 0.045 |
| 100 | 0.709 | 0.045 |

In [ ]:
k = 5
skf = StratifiedKFold(
    n_splits=k,
    shuffle=True,
    random_state=42
)

C_values = [0.01, 0.1, 1, 10, 100]
cv_results = []

for C in C_values:
    fold_scores = []
    for train_index, val_index in skf.split(X_train, y_train):
        X_fold_train = X_train.iloc[train_index]
        X_fold_val = X_train.iloc[val_index]
        y_fold_train = y_train.iloc[train_index]
        y_fold_val = y_train.iloc[val_index]

        model = LogisticRegression(C=C, max_iter=1000)
        model.fit(X_fold_train, y_fold_train)

        y_fold_pred = model.predict(X_fold_val)
        score = accuracy_score(y_fold_val, y_fold_pred)
        fold_scores.append(score)

    mean_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)

    cv_results.append({
        "C": C,
        "Mean CV Accuracy": mean_score,
        "Std": std_score
    })

    print(
        f"C = {C} | "
        f"Mean Accuracy = {mean_score:.3f} | "
        f"Std = {std_score:.3f}"
    )

### 6.4 Create comparison table, 6.5 Select best C, and 6.6 Train final model

In [ ]:
results_df = pd.DataFrame(cv_results)

results_df["Mean CV Accuracy"] = results_df["Mean CV Accuracy"] * 100
results_df["Std"] = results_df["Std"] * 100

display(results_df)

best_index = results_df["Mean CV Accuracy"].idxmax()
best_C = results_df.loc[best_index, "C"]
print("Best C selected by 5-Fold CV:", best_C)

# 6.6 Train final model using ALL training data
final_model = LogisticRegression(
    C=best_C,
    max_iter=1000
)
final_model.fit(X_train, y_train)

# Predict on untouched test data
final_pred = final_model.predict(X_test)
final_accuracy = accuracy_score(y_test, final_pred)

print("Final Test Accuracy:", round(final_accuracy, 3))
print("Final Test Accuracy: {:.2f}%".format(final_accuracy * 100))

# 6.7 Compare baseline and final
print("\n===== Accuracy Comparison =====")
print("Baseline Test Accuracy: {:.2f}%".format(baseline_accuracy * 100))
print("K-Fold Selected Test Accuracy: {:.2f}%".format(final_accuracy * 100))
print("Difference: {:+.2f} percentage points".format((final_accuracy - baseline_accuracy) * 100))

### Task 4: K-Fold Model Selection
**Use the C table and final comparison to explain the role of K-Fold.**

1. **Which $C$ was selected and why?**
   - **$C = 0.1$** was selected because it achieved the highest mean cross-validation accuracy (**71.8%** vs. 70.9% for default $C=1.0$).

2. **What were the baseline and final test accuracies?**
   - Baseline Test Accuracy: **65.82%**
   - Final Test Accuracy: **68.35%**

3. **What was the change in percentage points?**
   - An increase of **+2.53 percentage points**.

4. **Was K-Fold itself an accuracy booster? What was its actual role?**
   - **No.** K-Fold is an evaluation/selection protocol, not a predictive algorithm or optimizer. Its role is to act as an objective, low-variance selector to identify optimal hyperparameters without touching or leaking the test set.

## 7. Part 5 – Final Model Evaluation
### 7.1 Final confusion matrix & 7.2 Final classification report

| Actual / Predicted | Fail | Pass |
| :--- | :--- | :--- |
| **Fail** | 7 | 19 |
| **Pass** | 6 | 47 |

| Class/Metric | Precision | Recall | F1 | Support |
| :--- | :--- | :--- | :--- | :--- |
| **Fail** | 0.54 | 0.27 | 0.36 | 26 |
| **Pass** | 0.71 | 0.89 | 0.79 | 53 |
| **Accuracy** | | | 0.68 | 79 |
| **Macro avg** | 0.63 | 0.58 | 0.57 | 79 |
| **Weighted avg** | 0.65 | 0.68 | 0.65 | 79 |

In [ ]:
cm_final = confusion_matrix(y_test, final_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_final,
    display_labels=["Fail", "Pass"]
)
disp.plot()
plt.title("Final Model - Test Confusion Matrix")
plt.show()

print(classification_report(
    y_test,
    final_pred,
    target_names=["Fail", "Pass"]
))

### Task 5: Final Model Interpretation
**Interpret the final model using its confusion matrix and classification report.**

1. **Identify final TP, TN, FP, and FN:**
   - True Positives (TP): **47**
   - True Negatives (TN): **7**
   - False Positives (FP): **19**
   - False Negatives (FN): **6**

2. **How many Pass students were correctly classified?**
   - **47 students** (TP).

3. **How many Fail students were incorrectly classified as Pass?**
   - **19 students** (FP).

4. **Compare Pass recall with Fail recall:**
   - Pass Recall is **0.89** (89%), whereas Fail Recall remains critically low at **0.27** (27%).

5. **Why can accuracy be 68.35% while Fail recall is only 0.27?**
   - Because passing students make up the majority (67.09% of the dataset and 53 of 79 test observations). Correctly predicting 47 out of 53 passing students dominates the numerator of the accuracy formula $\frac{TP+TN}{\text{Total}}$, masking the fact that 19 out of 26 failing students were missed.